# 1. ENVIRONMENT SETUP

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'
import sys
import warnings
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple, List, Optional, Dict
import json
import time
from datetime import datetime
import shutil

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator

import cv2
from PIL import Image

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU'))} GPU(s)")
print(f"GPU Details: {tf.config.list_physical_devices('GPU')}")

# 2. CONFIGURATION

In [ ]:
class Config:
    """Central configuration for the project"""
    

    BASE_DIR = Path(".")
    DATA_DIR = BASE_DIR / "Data"
    LABELS_CSV = BASE_DIR / "labels.csv"
    MODEL_DIR = BASE_DIR / "models"
    SPLIT_DIR = BASE_DIR / "split_data"  
    
    
    MODEL_DIR.mkdir(exist_ok=True)
    SPLIT_DIR.mkdir(exist_ok=True)
    
    # Data Parameters
    IMG_HEIGHT = 48
    IMG_WIDTH = 48
    IMG_CHANNELS = 3
    NUM_CLASSES = 6
    NUM_CLASSES_WITH_BG = 7  
    
    # Data Split Ratios
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    
    # Training Parameters
    BATCH_SIZE = 32
    EPOCHS = 10
    LEARNING_RATE = 0.001
    
    # Model Parameters
    MODEL_NAME = "traffic_sign_classifier_final"
    
    # Edge Deployment Parameters
    MAX_MODEL_SIZE_MB = 50
    TARGET_FPS = 15
    
    # Random Seeds for Reproducibility
    SEED = 42
    
    # Class Labels
    CLASS_NAMES = {
        0: "Speed limit (30km/h)",
        1: "Speed limit (60km/h)",
        2: "Stop",
        3: "Go straight",
        4: "Go Left",
        5: "Go Right",
        6: "Background/No Sign"  # Added for false positive reduction
    }
    
    # Class weights for imbalanced data (will be calculated)
    CLASS_WEIGHTS = None

config = Config()

# Set random seeds for reproducibility
def set_seeds(seed=42):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seeds(config.SEED)

# Configure GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✓ GPU memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"GPU configuration error: {e}")

# 3. DATA LOADING


In [ ]:
class DataManager:
    """Manage data loading and train/val/test splitting"""
    
    def __init__(self, config: Config):
        self.config = config
        self.all_images = []
        self.all_labels = []
        self.class_distribution = {}
        
    def load_all_data(self) -> Tuple[List, List]:
        """Load all images from Data directory"""
        print("=" * 50)
        print("Loading dataset from Data directory...")
        print("=" * 50)
        
        for class_id in range(self.config.NUM_CLASSES):
            class_dir = self.config.DATA_DIR / str(class_id)
            
            if not class_dir.exists():
                print(f"⚠ Warning: Directory {class_dir} not found")
                continue
            
            # Get all image files
            image_files = list(class_dir.glob("*.png")) + \
                         list(class_dir.glob("*.jpg")) + \
                         list(class_dir.glob("*.jpeg"))
            
            self.class_distribution[class_id] = len(image_files)
            
            print(f"\nClass {class_id} - {self.config.CLASS_NAMES[class_id]}:")
            print(f"  Found {len(image_files)} images")
            
            # Store image paths and labels
            for img_path in image_files:
                self.all_images.append(str(img_path))
                self.all_labels.append(class_id)
        
        print(f"\n✓ Total images loaded: {len(self.all_images)}")
        print(f"✓ Class distribution: {self.class_distribution}")
        
        return self.all_images, self.all_labels
    
    def split_data(self, images, labels):
        """Split data into train, validation, and test sets"""
        print("\n" + "=" * 50)
        print("Splitting data into train/val/test sets...")
        print("=" * 50)
        
        # Convert to numpy arrays for easier handling
        images = np.array(images)
        labels = np.array(labels)
        
        # First split: separate test set
        X_temp, X_test, y_temp, y_test = train_test_split(
            images, labels,
            test_size=self.config.TEST_RATIO,
            random_state=self.config.SEED,
            stratify=labels
        )
        
        # Second split: separate train and validation
        val_size = self.config.VAL_RATIO / (1 - self.config.TEST_RATIO)
        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=val_size,
            random_state=self.config.SEED,
            stratify=y_temp
        )
        
        print(f"✓ Training set: {len(X_train)} images")
        print(f"✓ Validation set: {len(X_val)} images")
        print(f"✓ Test set: {len(X_test)} images")
        
        # Print class distribution for each set
        print("\nClass distribution in each set:")
        for name, y in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
            dist = np.bincount(y, minlength=self.config.NUM_CLASSES)
            print(f"\n{name}:")
            for i, count in enumerate(dist):
                print(f"  Class {i} ({self.config.CLASS_NAMES[i]}): {count}")
        
        return X_train, X_val, X_test, y_train, y_val, y_test
    
    def load_images_as_arrays(self, image_paths, labels=None, augment_speed_limits=True):
        """Load images as numpy arrays"""
        images = []
        valid_labels = [] if labels is not None else None
        
        print(f"Loading {len(image_paths)} images...")
        
        for idx, img_path in enumerate(tqdm(image_paths)):
            try:
                # Load image
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (self.config.IMG_WIDTH, self.config.IMG_HEIGHT))
                images.append(img)
                
                if labels is not None:
                    valid_labels.append(labels[idx])
                    
            except Exception as e:
                print(f"Error loading {img_path}: {e}")
        
        images = np.array(images, dtype=np.uint8)
        
        if labels is not None:
            valid_labels = np.array(valid_labels, dtype=np.int32)
            
            # Augment speed limit classes if requested
            if augment_speed_limits:
                images, valid_labels = self.augment_speed_limit_classes(images, valid_labels)
            
            return images, valid_labels
        
        return images
    
    def augment_speed_limit_classes(self, images, labels, augment_factor=2):
        """Augment speed limit classes (0 and 1) for better detection"""
        print("\nAugmenting speed limit classes for better detection...")
        
        augmented_images = []
        augmented_labels = []
        
        speed_limit_classes = [0, 1]  # 30km/h and 60km/h
        
        for class_id in speed_limit_classes:
            class_indices = np.where(labels == class_id)[0]
            class_images = images[class_indices]
            
            print(f"  Augmenting class {class_id} ({len(class_images)} original images)...")
            
            for _ in range(augment_factor):
                for img in class_images:
                    # Apply augmentations
                    aug_img = self.apply_augmentation(img)
                    augmented_images.append(aug_img)
                    augmented_labels.append(class_id)
        
        # Combine original and augmented
        if augmented_images:
            all_images = np.concatenate([images, np.array(augmented_images)])
            all_labels = np.concatenate([labels, np.array(augmented_labels)])
            print(f"  Added {len(augmented_images)} augmented images")
            return all_images, all_labels
        
        return images, labels
    
    def apply_augmentation(self, img):
        """Apply random augmentation to an image"""
        aug_img = img.copy()
        
        # Random rotation
        if np.random.random() > 0.5:
            angle = np.random.uniform(-10, 10)
            center = (self.config.IMG_WIDTH // 2, self.config.IMG_HEIGHT // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            aug_img = cv2.warpAffine(aug_img, M, (self.config.IMG_WIDTH, self.config.IMG_HEIGHT))
        
        # Random brightness
        if np.random.random() > 0.5:
            brightness = np.random.uniform(0.8, 1.2)
            aug_img = cv2.convertScaleAbs(aug_img, alpha=brightness, beta=0)
        
        # Random noise
        if np.random.random() > 0.5:
            noise = np.random.normal(0, 5, aug_img.shape).astype(np.int16)
            aug_img = np.clip(aug_img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        
        return aug_img
    
    def create_background_samples(self, n_samples=500):
        """Create synthetic background samples for false positive reduction"""
        print(f"\nCreating {n_samples} background samples...")
        
        backgrounds = []
        
        for i in range(n_samples):
            if i % 4 == 0:
                # Random noise pattern
                bg = np.random.randint(50, 200, 
                                      (self.config.IMG_HEIGHT, self.config.IMG_WIDTH, 3), 
                                      dtype=np.uint8)
            elif i % 4 == 1:
                # Gradient pattern
                bg = np.zeros((self.config.IMG_HEIGHT, self.config.IMG_WIDTH, 3), dtype=np.uint8)
                for c in range(3):
                    bg[:, :, c] = np.linspace(np.random.randint(0, 128), 
                                             np.random.randint(128, 256), 
                                             self.config.IMG_HEIGHT)[:, np.newaxis]
            elif i % 4 == 2:
                # Blurred noise
                bg = np.random.randint(0, 256, 
                                      (self.config.IMG_HEIGHT, self.config.IMG_WIDTH, 3), 
                                      dtype=np.uint8)
                bg = cv2.GaussianBlur(bg, (7, 7), 0)
            else:
                # Solid color with variations
                color = np.random.randint(50, 200, 3)
                bg = np.full((self.config.IMG_HEIGHT, self.config.IMG_WIDTH, 3), color, dtype=np.uint8)
                noise = np.random.randint(-30, 30, bg.shape)
                bg = np.clip(bg.astype(np.int16) + noise, 0, 255).astype(np.uint8)
            
            backgrounds.append(bg)
        
        backgrounds = np.array(backgrounds, dtype=np.uint8)
        background_labels = np.full(n_samples, 6, dtype=np.int32)  # Class 6 for background
        
        return backgrounds, background_labels
    
    def visualize_distribution(self, labels, title="Class Distribution"):
        """Visualize class distribution"""
        plt.figure(figsize=(12, 5))
        
        # Count occurrences
        unique, counts = np.unique(labels, return_counts=True)
        
        # Bar plot
        plt.subplot(1, 2, 1)
        colors = plt.cm.viridis(np.linspace(0, 1, len(unique)))
        bars = plt.bar(unique, counts, color=colors)
        plt.xlabel('Class ID')
        plt.ylabel('Number of Samples')
        plt.title(title)
        plt.xticks(unique)
        
        # Add value labels on bars
        for bar, count in zip(bars, counts):
            plt.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{count}', ha='center', va='bottom')
        
        # Pie chart
        plt.subplot(1, 2, 2)
        labels_pie = [f"{self.config.CLASS_NAMES.get(i, f'Class {i}')}" for i in unique]
        plt.pie(counts, labels=labels_pie, autopct='%1.1f%%', colors=colors)
        plt.title('Class Distribution Percentage')
        
        plt.tight_layout()
        plt.show()

# Initialize data manager
data_manager = DataManager(config)

# Load all data
all_image_paths, all_labels = data_manager.load_all_data()

# Split data
X_train_paths, X_val_paths, X_test_paths, y_train, y_val, y_test = data_manager.split_data(
    all_image_paths, all_labels
)

# 4. LOAD AND PREPARE IMAGE DATA

In [ ]:
print("\n" + "=" * 50)
print("Loading and preparing image data...")
print("=" * 50)

# Load training images (with speed limit augmentation)
X_train, y_train = data_manager.load_images_as_arrays(X_train_paths, y_train, augment_speed_limits=True)

# Load validation images (no augmentation)
X_val, y_val = data_manager.load_images_as_arrays(X_val_paths, y_val, augment_speed_limits=False)

# Load test images (no augmentation)
X_test, y_test = data_manager.load_images_as_arrays(X_test_paths, y_test, augment_speed_limits=False)

# Add background samples to training data
background_samples, background_labels = data_manager.create_background_samples(n_samples=len(X_train) // 6)
X_train = np.concatenate([X_train, background_samples], axis=0)
y_train = np.concatenate([y_train, background_labels], axis=0)

# Add fewer background samples to validation
val_bg_samples, val_bg_labels = data_manager.create_background_samples(n_samples=len(X_val) // 10)
X_val = np.concatenate([X_val, val_bg_samples], axis=0)
y_val = np.concatenate([y_val, val_bg_labels], axis=0)

# Shuffle training data
shuffle_idx = np.random.permutation(len(X_train))
X_train = X_train[shuffle_idx]
y_train = y_train[shuffle_idx]

print(f"\n✓ Final dataset sizes:")
print(f"  Training: {X_train.shape}")
print(f"  Validation: {X_val.shape}")
print(f"  Test: {X_test.shape}")

# Visualize final distributions
data_manager.visualize_distribution(y_train, "Training Set Distribution (with augmentation)")
data_manager.visualize_distribution(y_val, "Validation Set Distribution")
data_manager.visualize_distribution(y_test, "Test Set Distribution")


# 5. DATA AUGMENTATION


In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest',
    horizontal_flip=False  # Don't flip traffic signs
)

# Only normalization for validation and test
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Visualize augmentations
def visualize_augmentations(X_sample, y_sample, datagen, n_samples=8):
    """Visualize augmented samples"""
    fig, axes = plt.subplots(3, n_samples, figsize=(20, 9))
    
    indices = np.random.choice(len(X_sample), n_samples, replace=False)
    
    for i, idx in enumerate(indices):
        # Original
        axes[0, i].imshow(X_sample[idx])
        axes[0, i].set_title(f'Original\n{config.CLASS_NAMES[y_sample[idx]]}', fontsize=9)
        axes[0, i].axis('off')
        
        # Generate augmented versions
        img_batch = np.expand_dims(X_sample[idx], axis=0)
        
        # First augmentation
        aug_iter = datagen.flow(img_batch, batch_size=1)
        augmented1 = next(aug_iter)[0]
        axes[1, i].imshow(augmented1)
        axes[1, i].set_title('Augmented 1', fontsize=9)
        axes[1, i].axis('off')
        
        # Second augmentation
        augmented2 = next(aug_iter)[0]
        axes[2, i].imshow(augmented2)
        axes[2, i].set_title('Augmented 2', fontsize=9)
        axes[2, i].axis('off')
    
    plt.suptitle('Data Augmentation Examples', fontsize=14)
    plt.tight_layout()
    plt.show()

print("\nData Augmentation Examples:")
visualize_augmentations(X_train, y_train, train_datagen)

# 6. MODEL ARCHITECTURE


In [ ]:
def build_traffic_sign_model(input_shape=(48, 48, 3), num_classes=7):
    """
    Build CNN model for traffic sign classification
    No Lambda layers to avoid serialization issues
    """
    
    model = models.Sequential([
        # Input
        layers.Input(shape=input_shape),
        
        # Block 1: Initial feature extraction
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2: Deeper features
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3: Complex features
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Dense layers
        layers.Flatten(),
        layers.Dense(512),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.5),
        
        layers.Dense(128),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Dropout(0.3),
        
        # Output layer
        layers.Dense(num_classes, activation='softmax')
    ], name='traffic_sign_classifier')
    
    return model

# Build the model
print("\n" + "=" * 50)
print("Building Model Architecture...")
print("=" * 50)

model = build_traffic_sign_model(
    input_shape=(config.IMG_HEIGHT, config.IMG_WIDTH, config.IMG_CHANNELS),
    num_classes=config.NUM_CLASSES_WITH_BG
)

# Model summary
model.summary()

# Calculate model size
param_count = model.count_params()
model_size_mb = (param_count * 4) / (1024 * 1024)  # Assuming float32

print(f"\n📊 Model Statistics:")
print(f"  Total parameters: {param_count:,}")
print(f"  Estimated size: {model_size_mb:.2f} MB")

if model_size_mb < config.MAX_MODEL_SIZE_MB:
    print(f"  ✓ Model size within limit ({config.MAX_MODEL_SIZE_MB} MB)")
else:
    print(f"  ⚠ Model size exceeds limit ({config.MAX_MODEL_SIZE_MB} MB)")

# 7. TRAINING CONFIGURATION


In [ ]:
print("\nCalculating class weights...")
class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights_dict = dict(enumerate(class_weights))

print("Class weights:")
for class_id, weight in class_weights_dict.items():
    class_name = config.CLASS_NAMES.get(class_id, f"Class {class_id}")
    print(f"  {class_name}: {weight:.3f}")

# Compile model
model.compile(
    optimizer=optimizers.Adam(learning_rate=config.LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model compiled successfully")

# Define callbacks
callbacks_list = [
    # Save best model
    callbacks.ModelCheckpoint(
        filepath=str(config.MODEL_DIR / f"{config.MODEL_NAME}_best.h5"),
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Early stopping
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Reduce learning rate on plateau
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),
    
    # TensorBoard logging
    callbacks.TensorBoard(
        log_dir=str(config.MODEL_DIR / "logs" / datetime.now().strftime("%Y%m%d-%H%M%S")),
        histogram_freq=0,
        write_graph=True,
        update_freq='epoch'
    )
]


# 8. MODEL TRAINING


In [ ]:
print("\n" + "=" * 50)
print("Starting Model Training")
print("=" * 50)

# Calculate steps
steps_per_epoch = len(X_train) // config.BATCH_SIZE
validation_steps = len(X_val) // config.BATCH_SIZE

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Validation steps: {validation_steps}")
print(f"Batch size: {config.BATCH_SIZE}")

# Start training
start_time = time.time()

history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=config.BATCH_SIZE),
    steps_per_epoch=steps_per_epoch,
    epochs=config.EPOCHS,
    validation_data=val_datagen.flow(X_val, y_val, batch_size=config.BATCH_SIZE),
    validation_steps=validation_steps,
    class_weight=class_weights_dict,
    callbacks=callbacks_list,
    verbose=1
)

training_time = time.time() - start_time
print(f"\n✓ Training completed in {training_time/60:.2f} minutes")

# 9. TRAINING VISUALIZATION


In [ ]:
def plot_training_history(history):
    """Visualize training history"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Accuracy plot
    axes[0, 0].plot(history.history['accuracy'], label='Training', linewidth=2)
    axes[0, 0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
    axes[0, 0].axhline(y=0.9, color='r', linestyle='--', alpha=0.5, label='Target (90%)')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].set_title('Model Accuracy Over Time')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Loss plot
    axes[0, 1].plot(history.history['loss'], label='Training', linewidth=2)
    axes[0, 1].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].set_title('Model Loss Over Time')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Learning rate if available
    if 'lr' in history.history:
        axes[1, 0].plot(history.history['lr'], linewidth=2, color='green')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Learning Rate')
        axes[1, 0].set_title('Learning Rate Schedule')
        axes[1, 0].set_yscale('log')
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].axis('off')
    
    # Summary statistics
    axes[1, 1].axis('off')
    
    best_val_acc = max(history.history['val_accuracy'])
    best_val_loss = min(history.history['val_loss'])
    final_val_acc = history.history['val_accuracy'][-1]
    final_val_loss = history.history['val_loss'][-1]
    
    summary_text = f"""
    Training Summary
    ================
    
    Best Validation:
      Accuracy: {best_val_acc*100:.2f}%
      Loss: {best_val_loss:.4f}
    
    Final Validation:
      Accuracy: {final_val_acc*100:.2f}%
      Loss: {final_val_loss:.4f}
    
    Total Epochs: {len(history.history['accuracy'])}
    
    Target Achieved: {'✓ Yes' if final_val_acc >= 0.9 else '✗ No'}
    """
    
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=12, 
                   family='monospace', verticalalignment='center')
    
    plt.suptitle('Training History Analysis', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    return best_val_acc, final_val_acc

best_acc, final_acc = plot_training_history(history)


# 10. MODEL EVALUATION


In [ ]:
def evaluate_model(model, X, y, class_names, data_name="Test"):
    """Comprehensive model evaluation"""
    
    # Normalize data if needed
    if X.dtype == np.uint8:
        X_norm = X.astype(np.float32) / 255.0
    else:
        X_norm = X
    
    print(f"\n{'=' * 50}")
    print(f"Evaluating on {data_name} Set")
    print('=' * 50)
    
    # Get predictions
    y_pred_proba = model.predict(X_norm, verbose=0)
    y_pred = np.argmax(y_pred_proba, axis=1)
    
    # Calculate accuracy
    accuracy = accuracy_score(y, y_pred)
    print(f"\nOverall Accuracy: {accuracy*100:.2f}%")
    
    # Classification report
    print("\nDetailed Classification Report:")
    print("=" * 50)
    target_names = [class_names[i] for i in sorted(np.unique(y))]
    print(classification_report(y, y_pred, target_names=target_names))
    
    # Confusion matrix
    cm = confusion_matrix(y, y_pred)
    
    # Plot confusion matrix
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=target_names,
               yticklabels=target_names)
    plt.title(f'Confusion Matrix - {data_name} Set')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    return y_pred, accuracy, cm

# Evaluate on validation set
print("\nValidation Set Evaluation:")
y_val_pred, val_accuracy, val_cm = evaluate_model(
    model, X_val, y_val, config.CLASS_NAMES, "Validation"
)

# Evaluate on test set
print("\nTest Set Evaluation:")
y_test_pred, test_accuracy, test_cm = evaluate_model(
    model, X_test, y_test, config.CLASS_NAMES, "Test"
)

# 11. MISCLASSIFICATION ANALYSIS


In [ ]:
def analyze_misclassifications(X, y_true, y_pred, class_names, n_samples=10):
    """Analyze and visualize misclassified samples"""
    
    # Find misclassified indices
    misclassified_idx = np.where(y_true != y_pred)[0]
    
    if len(misclassified_idx) == 0:
        print("No misclassifications found!")
        return
    
    print(f"\nTotal misclassifications: {len(misclassified_idx)} out of {len(y_true)}")
    print(f"Misclassification rate: {len(misclassified_idx)/len(y_true)*100:.2f}%")
    
    # Analyze misclassification patterns
    misclass_matrix = np.zeros((len(class_names), len(class_names)))
    for idx in misclassified_idx:
        true_class = y_true[idx]
        pred_class = y_pred[idx]
        misclass_matrix[true_class, pred_class] += 1
    
    # Most common misclassifications
    print("\nMost common misclassifications:")
    misclass_pairs = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and misclass_matrix[i, j] > 0:
                misclass_pairs.append((misclass_matrix[i, j], i, j))
    
    misclass_pairs.sort(reverse=True)
    for count, true_idx, pred_idx in misclass_pairs[:5]:
        print(f"  {class_names[true_idx]} → {class_names[pred_idx]}: {int(count)} times")
    
    # Visualize some misclassified samples
    n_show = min(n_samples, len(misclassified_idx))
    sample_idx = np.random.choice(misclassified_idx, n_show, replace=False)
    
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.ravel()
    
    for i, idx in enumerate(sample_idx[:10]):
        if X[idx].dtype == np.uint8:
            img = X[idx]
        else:
            img = (X[idx] * 255).astype(np.uint8)
            
        axes[i].imshow(img)
        true_label = class_names[y_true[idx]]
        pred_label = class_names[y_pred[idx]]
        axes[i].set_title(f"True: {true_label}\nPred: {pred_label}", fontsize=9)
        axes[i].axis('off')
    
    plt.suptitle('Misclassified Samples', fontsize=14)
    plt.tight_layout()
    plt.show()

# Analyze misclassifications
analyze_misclassifications(X_test, y_test, y_test_pred, config.CLASS_NAMES)

# 12. MODEL OPTIMIZATION FOR EDGE DEPLOYMENT


In [ ]:
def convert_to_tflite(model, X_sample, save_path):
    """Convert model to TensorFlow Lite format"""
    
    print("\n" + "=" * 50)
    print("Converting to TensorFlow Lite")
    print("=" * 50)
    
    # Create converter
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # Set optimization
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    
    # Representative dataset for quantization
    def representative_dataset():
        for i in range(min(100, len(X_sample))):
            # Ensure proper format
            data = X_sample[i:i+1].astype(np.float32)
            if data.max() > 1.0:
                data = data / 255.0
            yield [data]
    
    converter.representative_dataset = representative_dataset
    
    # Optional: Full integer quantization for maximum size reduction
    # converter.target_spec.supported_types = [tf.int8]
    
    try:
        # Convert the model
        tflite_model = converter.convert()
        
        # Save the model
        with open(save_path, 'wb') as f:
            f.write(tflite_model)
        
        # Calculate size
        size_mb = len(tflite_model) / (1024 * 1024)
        
        print(f"✓ TFLite model saved: {save_path}")
        print(f"  Size: {size_mb:.2f} MB")
        
        if size_mb < config.MAX_MODEL_SIZE_MB:
            print(f"  ✓ Within size limit ({config.MAX_MODEL_SIZE_MB} MB)")
        else:
            print(f"  ⚠ Exceeds size limit ({config.MAX_MODEL_SIZE_MB} MB)")
        
        return tflite_model, size_mb
        
    except Exception as e:
        print(f"Error during conversion: {e}")
        return None, 0

# Convert to TFLite
tflite_path = config.MODEL_DIR / f"{config.MODEL_NAME}.tflite"
tflite_model, tflite_size = convert_to_tflite(model, X_train[:100], tflite_path)


# 13. TEST TFLITE MODEL

In [ ]:
def test_tflite_model(tflite_path, X_test, y_test, n_samples=100):
    """Test TFLite model performance"""
    
    print("\n" + "=" * 50)
    print("Testing TFLite Model")
    print("=" * 50)
    
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    
    # Get input and output details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    print(f"Input shape: {input_details[0]['shape']}")
    print(f"Output shape: {output_details[0]['shape']}")
    
    # Test on samples
    correct = 0
    total_time = 0
    n_test = min(n_samples, len(X_test))
    
    for i in range(n_test):
        # Prepare input
        input_data = X_test[i:i+1].astype(np.float32)
        if input_data.max() > 1.0:
            input_data = input_data / 255.0
        
        # Set input tensor
        interpreter.set_tensor(input_details[0]['index'], input_data)
        
        # Run inference
        start_time = time.time()
        interpreter.invoke()
        inference_time = time.time() - start_time
        total_time += inference_time
        
        # Get output
        output_data = interpreter.get_tensor(output_details[0]['index'])
        predicted = np.argmax(output_data[0])
        
        if predicted == y_test[i]:
            correct += 1
    
    # Calculate metrics
    accuracy = (correct / n_test) * 100
    avg_inference_time = (total_time / n_test) * 1000  # ms
    estimated_fps = 1000 / avg_inference_time
    
    print(f"\n✓ TFLite Model Performance:")
    print(f"  Accuracy: {accuracy:.2f}%")
    print(f"  Avg Inference Time: {avg_inference_time:.2f} ms")
    print(f"  Estimated FPS: {estimated_fps:.1f}")
    
    if estimated_fps >= config.TARGET_FPS:
        print(f"  ✓ Meets target FPS ({config.TARGET_FPS})")
    else:
        print(f"  ⚠ Below target FPS ({config.TARGET_FPS})")
    
    return accuracy, avg_inference_time, estimated_fps

# Test TFLite model
if tflite_model:
    tflite_accuracy, inference_time, fps = test_tflite_model(
        tflite_path, X_test, y_test, n_samples=100
    )
